# tp-01 データセット構築と分割・分布検証

**実行場所: ホスト PC（stg）。** 先に tp-00 を通すこと。

1. LayerA 母表を構築（学習対象年 2020〜2025 のみ。2026 年は使わない）
2. 分割ルール（2020-23 train / 2024 valid / 2025 test）と、1 年単位の expanding window CV（purge 7 日 + embargo 3 日）を適用
3. 季節・トレンド・分布差の検証（月別／馬場／場／芝ダート、adversarial validation）
4. 結果を MLflow（実験 `keiba-tp-dataset`）と `data/local/modeling/training_pipeline/eda/` に保存

分割ルールは `src/pipeline/models/dataset_split.py` と `time_split.py`。ここでは再実装しない。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# .env から接続先の 2 キーだけを読む（他のキーは触らない）
_env = REPO_ROOT / ".env"
if _env.exists():
    for line in _env.read_text(encoding="utf-8").splitlines():
        k, _, v = line.partition("=")
        if k.strip() in ("KEIBA_ENV", "MLFLOW_TRACKING_URI") and k.strip() not in os.environ:
            os.environ[k.strip()] = v.strip().strip('"').strip("'")
print("REPO_ROOT:", REPO_ROOT, "| KEIBA_ENV:", os.environ.get("KEIBA_ENV"))

In [ ]:
import json, subprocess
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models.dataset_split import DEFAULT_MANIFEST, assign_split_column, load_manifest
from src.pipeline.models.time_split import TRAINING_YEARS, restrict_to_training_years, yearly_expanding_folds
from src.pipeline.models.layer_a_dataset import build_layer_a_dataframe

REBUILD = False  # True でキャッシュを使わず母表を作り直す
WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
EDA = WORK / "eda"
EDA.mkdir(parents=True, exist_ok=True)
CACHE = WORK / "layer_a.parquet"

MLFLOW_URI = require_tracking_server()
mlflow.set_experiment("keiba-tp-dataset")
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
manifest = load_manifest()
print("MLflow:", MLFLOW_URI, "| git:", GIT_COMMIT, "| manifest version:", manifest.get("version"))

In [ ]:
import gc
import pyarrow.parquet as pq
from src.pipeline.models.memory_utils import downcast_frame, frame_mb, read_parquet_columns, rss_mb

ID_KEEP = ["race_id", "horse_id", "horse_number", "date"]  # 型を変えない列
DIAG_COLS = ["race_id", "horse_number", "date", "finish_position",
             "venue_code", "surface", "distance", "track_condition", "grade", "race_class", "weather"]

if REBUILD or not CACHE.exists() or "has_speed_index" not in pq.read_schema(CACHE).names:  # 旧版キャッシュ（半数欠落）は作り直す
    full = build_layer_a_dataframe(years=list(TRAINING_YEARS), jra_only=True)
    full = downcast_frame(full, keep=ID_KEEP)  # float32 / 整数縮小 / category（木モデルの結果は実質不変）
    full.to_parquet(CACHE, index=False)
    print(f"母表を構築して保存: {CACHE} ({full.shape[1]} 列, {frame_mb(full):.0f} MB)")
    del full
    gc.collect()

# 診断に必要な列だけを polars で読む（数百列の母表を丸ごとメモリに載せない）
cols = [c for c in DIAG_COLS if c in pq.read_schema(CACHE).names]
df = read_parquet_columns(CACHE, cols)
print(f"診断用に読込: {len(cols)} 列, {frame_mb(df):.0f} MB | プロセス常駐 {rss_mb():.0f} MB")

n_before = len(df)
df = restrict_to_training_years(df)
assert len(df) == n_before, "学習対象外の年の行が混入していました"

df["race_date"] = pd.to_datetime(df["date"].astype(str), errors="coerce")
assert df["race_date"].notna().all(), "race_date に変換できない行があります"
df["is_win"] = (df["finish_position"] == 1).astype(int)
df["is_top2"] = (df["finish_position"] <= 2).astype(int)
df["is_top3"] = (df["finish_position"] <= 3).astype(int)
df["month"] = df["race_date"].dt.month
df["year"] = df["race_date"].dt.year
print(df.shape, "| 期間:", df["race_date"].min().date(), "〜", df["race_date"].max().date())
print("開催年別レース数:\n", df.groupby("year")["race_id"].nunique())

In [ ]:
df["split"] = assign_split_column(df, manifest)
assert (df["split"] != "unknown").all(), "split が割り当てられない行があります"

# レース単位で完全分離されていること
per_race = df.groupby("race_id")["split"].nunique()
assert (per_race == 1).all(), "同一レースが複数の split に跨っています"

split_summary = df.groupby("split").agg(
    races=("race_id", "nunique"), rows=("race_id", "size"),
    date_from=("race_date", "min"), date_to=("race_date", "max"),
).loc[["train", "valid", "test"]]
display(split_summary)

folds = list(yearly_expanding_folds(df, date_col="race_date"))
fold_summary = pd.DataFrame([{
    "fold": f.fold_id, "n_train": len(f.train_index), "n_valid": len(f.valid_index),
    "train_end(含まず)": f.train_end.date(), "valid_start": f.valid_start.date(),
} for f in folds])
display(fold_summary)

## 季節・トレンドの分布確認

評価期間が特定の季節・馬場・場に偏っていないか、年次ドリフトがないかを確認する。

In [ ]:
def share(frame, col):
    t = frame.groupby(["split", col])["race_id"].nunique().unstack("split").fillna(0)
    return (t / t.sum()).round(3)

month_share = share(df, "month")
display(month_share)

seg_cols = [c for c in ("surface", "venue_code", "track_condition") if c in df.columns]
for c in seg_cols:
    print(f"--- {c} の split 別構成比 ---")
    display(share(df, c))

if "track_condition" in df.columns:
    print("--- 馬場状態の年次トレンド ---")
    display(df.groupby("year")["track_condition"].value_counts(normalize=True).unstack().round(3))

df["field_size"] = df.groupby("race_id")["horse_number"].transform("size")
trend = df.groupby("year").agg(win_rate_by_field=("is_win", "mean"), field_size=("field_size", "mean"))
display(trend.round(3))

## adversarial validation

train と valid/test を見分けられるか（AUC）。0.5 に近いほど同分布。月を含む場合と含まない場合を比べ、季節偏りの影響を見る。

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

RACE_ATTRS = [c for c in ("venue_code", "surface", "distance", "track_condition", "field_size", "grade", "race_class", "weather") if c in df.columns]


def adversarial_auc(a, b, cols, n_splits=5, seed=42):
    x = pd.concat([a[cols + ["race_id"]], b[cols + ["race_id"]]], ignore_index=True)
    y = np.r_[np.zeros(len(a)), np.ones(len(b))]
    groups = x.pop("race_id")
    for c in x.columns:
        if x[c].dtype == object:
            x[c] = x[c].astype("category")
    oof = np.zeros(len(x))
    for tr, va in GroupKFold(n_splits=n_splits).split(x, y, groups):
        m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, random_state=seed, verbose=-1)
        m.fit(x.iloc[tr], y[tr])
        oof[va] = m.predict_proba(x.iloc[va])[:, 1]
    imp = pd.Series(m.feature_importances_, index=x.columns).sort_values(ascending=False)
    return roc_auc_score(y, oof), imp


tr, va, te = (df[df["split"] == s] for s in ("train", "valid", "test"))
adv = {}
for name, (a, b) in {"train_vs_valid": (tr, va), "train_vs_test": (tr, te), "valid_vs_test": (va, te)}.items():
    auc_nm, imp_nm = adversarial_auc(a, b, RACE_ATTRS)
    auc_m, imp_m = adversarial_auc(a, b, RACE_ATTRS + ["month"])
    adv[name] = {"auc_without_month": round(auc_nm, 4), "auc_with_month": round(auc_m, 4),
                 "top_features_without_month": imp_nm.head(5).to_dict()}
adv_df = pd.DataFrame({k: {"AUC(月なし)": v["auc_without_month"], "AUC(月あり)": v["auc_with_month"]} for k, v in adv.items()}).T
display(adv_df)

In [ ]:
profile = {
    "git_commit": GIT_COMMIT,
    "split_manifest_version": manifest.get("version"),
    "training_years": list(TRAINING_YEARS),
    "rows": int(len(df)), "races": int(df["race_id"].nunique()),
    "split": {k: {"races": int(v["races"]), "rows": int(v["rows"]),
                  "date_from": str(v["date_from"].date()), "date_to": str(v["date_to"].date())}
              for k, v in split_summary.iterrows()},
    "adversarial_validation": adv,
}
(EDA / "dataset_profile.json").write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding="utf-8")
month_share.to_csv(EDA / "month_share_by_split.csv")
fold_summary.to_csv(EDA / "cv_folds.csv", index=False)
label_balance = df.groupby("split")[["is_win", "is_top2", "is_top3"]].mean().round(4)
label_balance.to_json(EDA / "label_balance.json")

with mlflow.start_run(run_name="tp-01-dataset-split", tags={
    "notebook": "tp-01", "git_commit": GIT_COMMIT, "split_manifest_version": str(manifest.get("version")),
    "training_years": ",".join(TRAINING_YEARS),
}):
    mlflow.log_params({"rows": len(df), "races": df["race_id"].nunique(), "purge_days": 7, "embargo_days": 3,
                       "race_attr_cols": ",".join(RACE_ATTRS)})
    for k, v in adv.items():
        mlflow.log_metric(f"adv_auc_{k}_no_month", v["auc_without_month"])
        mlflow.log_metric(f"adv_auc_{k}_with_month", v["auc_with_month"])
    mlflow.log_artifacts(str(EDA), "eda")
print("保存:", EDA)
display(label_balance)